# Getting Started with Fine-Tuning Moshi 7B

This notebook shows you a simple example of how to LoRA finetune Moshi 7B. You can run this notebook in Google Colab using a A100 GPU.

<a target="_blank" href="https://colab.research.google.com/github//kyutai-labs/moshi-finetune/blob/main/tutorials/moshi_finetune.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Check out `moshi-finetune` Github repo to learn more: https://github.com/kyutai-labs/moshi-finetune/


## Installation

Clone the `moshi-finetune` repo:


In [ ]:
%cd /content/
!git clone https://github.com/kyutai-labs/moshi-finetune.git

/content
Cloning into 'moshi-finetune'...
remote: Enumerating objects: 253, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 253 (delta 87), reused 84 (delta 83), pack-reused 120 (from 1)
Receiving objects: 100% (253/253), 623.90 KiB | 5.07 MiB/s, done.
Resolving deltas: 100% (148/148), done.


Install all required dependencies:


In [ ]:
# Освобождаем место на диске перед установкой зависимостей
import shutil
from pathlib import Path
import os

# Удаляем все оставшиеся файлы parquet для освобождения места
parquet_dir = Path("/content/data/speech-ru/data")
if parquet_dir.exists():
    print("Очистка исходных parquet архивов для освобождения диска...")
    shutil.rmtree(parquet_dir)
    print("Диск успешно очищен!")

# Проверяем наличие репозитория moshi-finetune, если его нет — клонируем
repo_path = Path("/content/moshi-finetune")
if not repo_path.exists():
    print("Репозиторий moshi-finetune не найден. Клонируем...")
    !git clone https://github.com/kyutai-labs/moshi-finetune.git /content/moshi-finetune

# Напрямую ставим нужные версии, чтобы избежать конфликтов pip
!pip install "sphn>=0.2.0,<0.3.0"
!pip install moshi
!cd /content/moshi-finetune && pip install --no-deps .
# Доустанавливаем недостающие зависимости
!pip install fire simple-parsing pyyaml safetensors tqdm huggingface_hub sounddevice

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.2/507.2 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 944.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19

## Prepare dataset


In [ ]:
from pathlib import Path
from huggingface_hub import list_repo_files, hf_hub_download

local_dir = Path("/content/data/speech-ru")
local_dir.mkdir(parents=True, exist_ok=True)

# Получаем все файлы репозитория
files = list_repo_files(repo_id="Sinoosoida/SpeechRu", repo_type="dataset")

# Ограничение 50 ГБ
max_bytes = 50 * 1024 * 1024 * 1024
current_bytes = 0

for f in local_dir.glob('**/*'):
    if f.is_file():
        current_bytes += f.stat().st_size

# Отбираем файлы данных (parquet)
parquet_files = [f for f in files if f.endswith('.parquet')]

for f in parquet_files:
    if current_bytes >= max_bytes:
        print(f"Достигнут лимит 50 ГБ (Текущий размер: {current_bytes / (1024**3):.2f} ГБ).")
        break

    print(f"Загрузка: {f}")
    downloaded_path = hf_hub_download(repo_id="Sinoosoida/SpeechRu", filename=f, repo_type="dataset", local_dir=local_dir)
    current_bytes += Path(downloaded_path).stat().st_size
    print(f"Скачано {current_bytes / (1024**3):.2f} GB / 50 GB")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Загрузка: data/000/train-00000.parquet


data/000/train-00000.parquet:   0%|          | 0.00/149M [00:00<?, ?B/s]

Скачано 0.14 GB / 50 GB
Загрузка: data/000/train-00001.parquet


data/000/train-00001.parquet:   0%|          | 0.00/148M [00:00<?, ?B/s]

...

Скачано 49.90 GB / 50 GB
Загрузка: data/000/train-00808.parquet


data/000/train-00808.parquet:   0%|          | 0.00/83.4M [00:00<?, ?B/s]

Скачано 49.98 GB / 50 GB
Загрузка: data/000/train-00809.parquet


data/000/train-00809.parquet:   0%|          | 0.00/108M [00:00<?, ?B/s]

Скачано 50.08 GB / 50 GB
Достигнут лимит 50 ГБ (Текущий размер: 50.08 ГБ).


In [ ]:
import pandas as pd
import json
from pathlib import Path
import glob

# Посмотрим на структуру одного из скачанных parquet файлов
parquet_files = glob.glob('/content/data/speech-ru/**/*.parquet', recursive=True)
if parquet_files:
    df_sample = pd.read_parquet(parquet_files[0])
    print("Колонки в датасете:", df_sample.columns.tolist())
    display(df_sample.head(2))
else:
    print("Файлы не найдены. Убедитесь в правильности пути.")

Колонки в датасете: ['audio', 'id', 'name', 'slug', 'audio_type', 'type', 'lang', 'duration_sec', 'audio_bytes', 'rating', 'rating_votes', 'rating_scores', 'read_count', 'reviews_count', 'citations_count', 'main_author_slug', 'main_author_name', 'main_actor_slug', 'main_actor_name', 'publisher_name', 'genres_names', 'tags_csv', 'series_name', 'subscription_id', 'is_purchased', 'adult', 'written_dt', 'updated_at', 'absolute_url', 'share_url', 'default_cover_path', 'preview_audio_url', 'annotation_plain', 'annotation_html', 'page_url', 'raw_metadata_json']


,audio,id,name,slug,audio_type,type,lang,duration_sec,audio_bytes,rating,...,written_dt,updated_at,absolute_url,share_url,default_cover_path,preview_audio_url,annotation_plain,annotation_html,page_url,raw_metadata_json
0,{'bytes': b'\xff\xfb\xe0d\x00\x00\x00\x00\x00\...,737677,Токсичная коммуникация,toksichnaya-kommunikaciya,podcast,audio,ru,2805,112187558,5.0,...,2021-11-01,2024-12-25T14:58:27.155664,/author/veronika-sidorova/toksichnaya-kommunik...,https://mybook.ru/author/veronika-sidorova/tok...,book_covers/3b/12/3b12e858-cdf2-4434-9ea6-4cc7...,/api/audiofiles/953174/file.mp3,Сегодня в гостях у Вероники Сидоровой - Михаил...,<p>Сегодня в гостях у Вероники Сидоровой - Мих...,/author/veronika-sidorova/toksichnaya-kommunik...,"{""id"": 737677, ""resource_uri"": ""/api/v1/books/..."
1,{'bytes': b'ID3\x04\x00\x00\x00\x00\x02=TXXX\x...,869790,Какие продукты заставляют есть больше и набира...,kakie-produkty-zastavlyayut-est-bolshe-i-nabir...,podcast,audio,ru,457,7312104,5.0,...,2022-10-10,2024-12-25T18:37:53.761925,/author/lajfhaker/kakie-produkty-zastavlyayut-...,https://mybook.ru/author/lajfhaker/kakie-produ...,book_covers/60/42/6042e2f5-31ff-42d8-ac83-ba29...,/api/audiofiles/1329223/file.mp3,"Учёные уверены, что в наборе веса виновата обр...","<p>Учёные уверены, что в наборе веса виновата ...",/author/lajfhaker/kakie-produkty-zastavlyayut-...,"{""id"": 869790, ""resource_uri"": ""/api/v1/books/..."


In [ ]:
import os
import pandas as pd
import json
from pathlib import Path
import glob

audio_out_dir = Path("/content/data/speech-ru/audio_files")
audio_out_dir.mkdir(parents=True, exist_ok=True)

jsonl_records = []

# Ищем parquet файлы
parquet_files = glob.glob("/content/data/speech-ru/**/*.parquet", recursive=True)
parquet_files = list(set(parquet_files))

count = 0
print(f"Найдено {len(parquet_files)} parquet файлов для обработки.")

# Лимитируем общее число аудиозаписей, чтобы не забить диск
max_audio_files = 15000

for p_file in sorted(parquet_files):
    if count >= max_audio_files:
        print("Достигнут лимит на количество извлеченных аудиофайлов. Прекращаем обработку.")
        break
    try:
        df = pd.read_parquet(p_file)
    except Exception as e:
        print(f"Ошибка чтения {p_file}: {e}")
        continue

    for idx, row in df.iterrows():
        if count >= max_audio_files:
            break
        try:
            audio_data = row.get('audio')
            text = row.get('annotation_plain', '')
            if not text:
                text = row.get('name', '')

            audio_bytes = None
            if isinstance(audio_data, dict):
                audio_bytes = audio_data.get('bytes')
            elif hasattr(audio_data, 'get'):
                audio_bytes = audio_data.get('bytes')
            elif isinstance(audio_data, bytes):
                audio_bytes = audio_data

            if audio_bytes is not None:
                file_name = f"audio_{count}.mp3"
                file_path = audio_out_dir / file_name

                with open(file_path, 'wb') as f:
                    f.write(audio_bytes)

                jsonl_records.append({
                    "audio_path": str(file_path),
                    "text": str(text)
                })
                count += 1
        except Exception as e:
            continue

    # Удаляем обработанный parquet файл, чтобы освободить ценное место на диске
    try:
        os.remove(p_file)
    except Exception as e:
        print(f"Не удалось удалить {p_file}: {e}")

jsonl_path = "/content/data/speech-ru/dataset.jsonl"
with open(jsonl_path, "w", encoding="utf-8") as f:
    for record in jsonl_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Успешно подготовлено {len(jsonl_records)} аудиозаписей. Файл разметки: {jsonl_path}")

Найдено 487 parquet файлов для обработки.
Успешно подготовлено 1520 аудиозаписей. Файл разметки: /content/data/speech-ru/dataset.jsonl


## Start training


In [ ]:
# these info is needed for training
import os

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
# define training configuration pointing to the prepared SpeechRu dataset
import yaml

config = """
# data
data:
  train_data: '/content/data/speech-ru/dataset.jsonl'
  eval_data: ''
  shuffle: true

# model
moshi_paths:
  hf_repo_id: "kyutai/moshiko-pytorch-bf16"

full_finetuning: false
lora:
  enable: true
  rank: 128
  scaling: 2.
  ft_embed: false

# training hyperparameters
first_codebook_weight_multiplier: 100.
text_padding_weight: .5

duration_sec: 100
batch_size: 1
max_steps: 300

gradient_checkpointing: true

# optim
optim:
  lr: 2.e-6
  weight_decay: 0.1
  pct_start: 0.05

# other
seed: 0
log_freq: 10
eval_freq: 1
do_eval: False
ckpt_freq: 10

save_adapters: True

run_dir: "/content/test"
"""

with open("/content/example.yaml", "w") as file:
    yaml.dump(yaml.safe_load(config), file)

In [ ]:
# make sure the run_dir has not been created before
# only run this when you ran torchrun previously and created the /content/test_ultra file
# ! rm -r /content/test

In [ ]:
# Очистка старой директории запуска и запуск обучения
import os
import shutil
from pathlib import Path

# 1. Принудительно удаляем старую директорию /content/test, чтобы избежать конфликта
run_dir = Path('/content/test')
if run_dir.exists():
    print(f"Удаляем старую директорию логов {run_dir} для предотвращения конфликта...")
    shutil.rmtree(run_dir)
    print("Успешно очищено!")

# 2. Убеждаемся, что права доступа на /tmp в порядке
os.makedirs('/tmp', exist_ok=True)
os.chmod('/tmp', 0o777)

# 3. Запускаем обучение
!cd /content/moshi-finetune && torchrun --nproc-per-node 1 -m train /content/example.yaml

I0000 00:00:1791358372.671301   16334 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-10-07 07:33:02 (UTC) - 0:00:16 - distributed - INFO - torch.cuda.device_count: 0
2026-10-07 07:33:02 (UTC) - 0:00:16 - distributed - INFO - CUDA_VISIBLE_DEVICES: 0
2026-10-07 07:33:02 (UTC) - 0:00:16 - distributed - INFO - local rank: 0
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/moshi-finetune/train.py", line 360, in <module>
    fire.Fire(train)
    ~~~~~~~~~^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/fire/core.py", line 135, in Fire
    component_trace = _Fire(component, args, parsed_flag_args, context, name)
  File "/usr/local/lib/python3.13/dist-packages/fire/core

In [ ]:
import subprocess
import glob

# Find the training script
scripts = glob.glob('/content/moshi-finetune/**/train.py', recursive=True)
print("Found training scripts:", scripts)

if scripts:
    train_script = scripts[0]
    # Run training directly using python to get the full traceback without torchrun hiding the error
    try:
        result = subprocess.run(
            ["python", train_script, "/content/example.yaml"],
            capture_output=True,
            text=True,
            check=True
        )
        print("Training completed successfully!")
    except subprocess.CalledProcessError as e:
        print("--- Training failed with exit code", e.returncode, "---")
        print("\n--- Standard Output ---")
        print(e.stdout)
        print("\n--- Standard Error ---")
        print(e.stderr)
else:
    print("Could not find train.py inside /content/moshi-finetune")

Found training scripts: ['/content/moshi-finetune/train.py']
--- Training failed with exit code 1 ---

--- Standard Output ---
2026-10-07 07:33:17 (UTC) - 0:00:08 - train - ERROR - PyTorch environment is not correctly initialized. This message should only be displayed when testing.


--- Standard Error ---
I0000 00:00:1791358392.610673   16434 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-10-07 07:33:17 (UTC) - 0:00:08 - train - ERROR - PyTorch environment is not correctly initialized. This message should only be displayed when testing.
Traceback (most recent call last):
  File "/content/moshi-finetune/train.py", line 360, in <module>
    fire.Fire(train)
    ~~~~~~~~~^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/fire/core.py", line 135, in Fire
    component_trac

## Inference

Once the model has been trained, inference can be run on the colab GPU too, and gradio can be used to tunnel the audio data from a local client to the notebook.

More details on how to set this up can be found in the [moshi readme](https://github.com/kyutai-labs/moshi?tab=readme-ov-file#python-pytorch).


In [ ]:
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 13.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
finetune 0.0.0 requires auditok==0.2, which is not installed.
finetune 0.0.0 requires submitit, which is not installed.
finetune 0.0.0 requires whisper_timestamped, which is not installed.
moshi 0.2.13 requires huggingface-hub<1.0.0,>=0.24, but you have huggingface-hub 1.33.0 which is incompatible.
finetune 0.0.0 requires sphn==0.1.12, but you have sphn 0.2.1 which is incompatible.
finetune 0.0.0 requires torch==2.6, but you have torch 2.9.1 which is incompatible.
transformers 5.18.0 requires safetensors>=0.8.0, but you have safetensors 0.7.0 whic

In [ ]:
!python -m moshi.server --gradio-tunnel --lora-weight=/content/test/checkpoints/checkpoint_000300/consolidated/lora.safetensors --config-path=/content/test/checkpoints/checkpoint_000300/consolidated/config.json

[Info] retrieving checkpoint
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.13/dist-packages/moshi/server.py", line 287, in <module>
    main()
    ~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/moshi/server.py", line 223, in main
    checkpoint_info = loaders.CheckpointInfo.from_hf_repo(
        args.hf_repo, args.moshi_weight, args.mimi_weight, args.tokenizer,
        lora_weights=args.lora_weight, config_path=args.config_path)
  File "/usr/local/lib/python3.13/dist-packages/moshi/models/loaders.py", line 223, in from_hf_repo
    raw_config = json.loads(Path(config_path).read_text())
                            ~~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/usr/lib/python3.13/pathlib/_local.py", line 546, in read_text
    return PathBase.read_text(self, encoding, errors, newline)
           ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/pyt